Audit BriTROC public files and EGA access

In [1]:
from pathlib import Path
import requests, json, hashlib, os, re, zipfile, gzip, io, csv
from datetime import datetime, timezone

OUT = Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane1_britroc')
OUT.mkdir(parents=True, exist_ok=True)
DATA = Path('/workspace/mechanism_fit_britroc')
DATA.mkdir(parents=True, exist_ok=True)

def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def save_json(name,obj):
    p=OUT/name; p.write_text(json.dumps(obj,indent=2,default=str)); return p

def get(url, **kw):
    at=ts()
    try:
        r=requests.get(url,timeout=60,**kw)
        return {'url':url,'timestamp':at,'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:4000]}, r
    except Exception as e:
        return {'url':url,'timestamp':at,'status_code':None,'error':type(e).__name__+': '+str(e)[:500]}, None

manifest={'lane':'BriTROC public + controlled EGA','started_at':ts(),'secret_presence':{'EGA_USERNAME':bool(os.getenv('EGA_USERNAME')),'EGA_PASSWORD':bool(os.getenv('EGA_PASSWORD')),'ega_private_key':Path('/run/secrets/ega_private_key').exists()},'sources':[],'files':[]}

# Official GitHub repository tree and selected public data/docs.
repo='BRITROC/britroc-1-HGSOC-landscape'
branch=None
for candidate in ['main','master']:
    rec,r=get(f'https://api.github.com/repos/{repo}/git/trees/{candidate}?recursive=1')
    save_json(f'github_tree_{candidate}_receipt.json',rec)
    if r is not None and r.status_code==200:
        tree=r.json().get('tree',[]); branch=candidate; break
if branch:
    save_json('github_tree.json',tree)
    manifest['sources'].append({'source':'GitHub repository tree','identifier':f'https://github.com/{repo}','timestamp':ts(),'access':'ACCESSED','items':len(tree),'receipt':'github_tree.json'})
    selected=[]
    for x in tree:
        if x.get('type')!='blob': continue
        path=x.get('path',''); low=path.lower(); size=x.get('size') or 0
        if any(k in low for k in ['clinical','manifest','dictionary','metadata','sample','patient','treatment','outcome','platinum','supplement']) or low.endswith(('.csv','.tsv','.xlsx','.xls','.txt','.json','.rds','.rdata','.md')):
            selected.append(x)
    for x in selected:
        path=x['path']; size=x.get('size') or 0
        entry={'source':'GitHub','repository_path':path,'declared_size':size,'url':f'https://raw.githubusercontent.com/{repo}/{branch}/{path}','access_timestamp':ts()}
        if size <= 50_000_000:
            rr=requests.get(entry['url'],timeout=120)
            entry['status_code']=rr.status_code
            if rr.status_code==200:
                dst=DATA/('github__'+path.replace('/','__')); dst.parent.mkdir(parents=True,exist_ok=True); dst.write_bytes(rr.content)
                entry.update({'local_path':str(dst),'bytes':dst.stat().st_size,'sha256':sha(dst)})
        manifest['files'].append(entry)

# Official Zenodo record and each listed public release file.
rec,r=get('https://zenodo.org/api/records/8304868')
save_json('zenodo_record_receipt.json',rec)
if r is not None and r.status_code==200:
    z=r.json(); save_json('zenodo_record.json',z)
    zfiles=z.get('files',[])
    manifest['sources'].append({'source':'Zenodo BriTROC preprocessed release','identifier':'10.5281/zenodo.8304868','timestamp':rec['timestamp'],'access':'ACCESSED','items':len(zfiles),'receipt':'zenodo_record.json'})
    for f in zfiles:
        url=(f.get('links') or {}).get('self') or (f.get('links') or {}).get('content')
        size=f.get('size') or 0; name=f.get('key') or f.get('filename')
        entry={'source':'Zenodo','name':name,'url':url,'declared_size':size,'zenodo_checksum':f.get('checksum'),'access_timestamp':ts()}
        # Download all public release files up to 500 MB; larger files remain fully enumerated with checksum.
        if url and size <= 500_000_000:
            rr=requests.get(url,timeout=600,stream=True)
            entry['status_code']=rr.status_code
            if rr.status_code==200:
                dst=DATA/('zenodo__'+Path(name).name)
                with open(dst,'wb') as fh:
                    for b in rr.iter_content(1024*1024):
                        if b: fh.write(b)
                entry.update({'local_path':str(dst),'bytes':dst.stat().st_size,'sha256':sha(dst)})
                if zipfile.is_zipfile(dst):
                    with zipfile.ZipFile(dst) as zz:
                        entry['zip_members']=[{'name':i.filename,'size':i.file_size,'crc':i.CRC} for i in zz.infolist()]
                        # Extract small metadata/tabular members only.
                        for i in zz.infolist():
                            low=i.filename.lower()
                            if not i.is_dir() and i.file_size <= 100_000_000 and (low.endswith(('.csv','.tsv','.txt','.xlsx','.xls','.json','.rds','.rdata','.md')) or any(k in low for k in ['clinical','meta','sample','patient','manifest','treatment'])):
                                target=DATA/'zenodo_extracted'/i.filename
                                target.parent.mkdir(parents=True,exist_ok=True)
                                with zz.open(i) as src, open(target,'wb') as out: out.write(src.read())
                                manifest['files'].append({'source':'Zenodo archive member','archive':name,'name':i.filename,'bytes':target.stat().st_size,'local_path':str(target),'sha256':sha(target),'access_timestamp':ts(),'status_code':200})
        manifest['files'].append(entry)
else:
    manifest['sources'].append({'source':'Zenodo BriTROC preprocessed release','identifier':'10.5281/zenodo.8304868','timestamp':rec['timestamp'],'access':'NOT_FOUND','receipt':'zenodo_record_receipt.json'})

# EGA public metadata + exact unauthenticated controlled file-list attempts.
for eid,kind in [('EGAS00001007292','study'),('EGAD00001011049','dataset'),('EGAD00001011058','dataset'),('EGAC00001000388','dac')]:
    endpoints=[f'https://metadata.ega-archive.org/{kind}s/{eid}',f'https://ega-archive.org/{kind}s/{eid}']
    attempts=[]
    for j,url in enumerate(endpoints):
        rec,r=get(url,allow_redirects=True); save_json(f'{eid}_public_{j}_receipt.json',rec); attempts.append(rec)
    manifest['sources'].append({'source':'EGA public metadata','identifier':eid,'kind':kind,'timestamp':attempts[0]['timestamp'],'attempts':attempts})

# Data API permission check without credentials; no credential was available in this worker.
for eid in ['EGAD00001011049','EGAD00001011058']:
    urls=[f'https://ega.ebi.ac.uk:8443/v2/datasets/{eid}/files',f'https://ega.ebi.ac.uk:8443/v2/datasets/{eid}']
    for j,url in enumerate(urls):
        rec,r=get(url,allow_redirects=False)
        save_json(f'{eid}_controlled_access_{j}_receipt.json',rec)
        manifest['sources'].append({'source':'EGA controlled Data API','identifier':eid,'timestamp':rec['timestamp'],'status_code':rec.get('status_code'),'access':'ACCESS_DENIED' if rec.get('status_code') in [401,403] or not manifest['secret_presence']['EGA_USERNAME'] else 'NOT_ATTEMPTED','receipt':f'{eid}_controlled_access_{j}_receipt.json','note':'No EGA download credentials or approved private key were present in the execution environment; public metadata does not establish authorization.'})

manifest['completed_at']=ts()
save_json('lane1_manifest.json',manifest)
print(json.dumps({'sources':len(manifest['sources']),'file_records':len(manifest['files']),'downloaded':sum(1 for f in manifest['files'] if f.get('sha256')),'zenodo_files':sum(1 for f in manifest['files'] if f.get('source')=='Zenodo'),'output':str(OUT)},indent=2))

{
  "sources": 10,
  "file_records": 2,
  "downloaded": 1,
  "zenodo_files": 1,
  "output": "/mnt/shared-workspace/shared/mechanism_fit_audit/lane1_britroc"
}


Download and hash full BriTROC public release

In [3]:
from pathlib import Path
import requests, hashlib, zipfile, json
from datetime import datetime, timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane1_britroc')
DATA=Path('/workspace/mechanism_fit_britroc'); DATA.mkdir(parents=True,exist_ok=True)
meta=json.loads((OUT/'zenodo_record.json').read_text())
f=meta['files'][0]; url=(f.get('links') or {}).get('self') or (f.get('links') or {}).get('content'); name=f.get('key')
dst=DATA/('zenodo__'+Path(name).name)
h=hashlib.sha256(); n=0
with requests.get(url,stream=True,timeout=1800) as r:
    r.raise_for_status()
    with open(dst,'wb') as fh:
        for chunk in r.iter_content(4*1024*1024):
            if chunk: fh.write(chunk); h.update(chunk); n+=len(chunk)
entry={'source':'Zenodo','name':name,'url':url,'declared_size':f.get('size'),'bytes':n,'sha256':h.hexdigest(),'access_timestamp':datetime.now(timezone.utc).isoformat().replace('+00:00','Z'),'status_code':200,'local_path':str(dst)}
with zipfile.ZipFile(dst) as z:
    entry['zip_members']=[{'name':i.filename,'size':i.file_size,'crc':i.CRC} for i in z.infolist()]
    for i in z.infolist():
        low=i.filename.lower()
        if not i.is_dir() and i.file_size<=300_000_000 and (low.endswith(('.csv','.tsv','.txt','.xlsx','.xls','.json','.rds','.rdata','.md')) or any(k in low for k in ['clinical','metadata','sample','patient','manifest','treatment','outcome'])):
            target=DATA/'zenodo_extracted'/i.filename; target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(i) as src, open(target,'wb') as out:
                hh=hashlib.sha256()
                for b in iter(lambda:src.read(1024*1024),b''): out.write(b); hh.update(b)
            entry.setdefault('extracted',[]).append({'name':i.filename,'bytes':i.file_size,'sha256':hh.hexdigest(),'local_path':str(target)})
(OUT/'zenodo_full_download.json').write_text(json.dumps(entry,indent=2))
print(json.dumps({'bytes':n,'sha256':entry['sha256'],'members':len(entry['zip_members']),'extracted':len(entry.get('extracted',[])),'first_members':entry['zip_members'][:20]},indent=2))

{
  "bytes": 1356068166,
  "sha256": "fad81b4e8a0298b26fafc6e91a1e83c99856cafb9942eddd9efca89b2db071af",
  "members": 1351,
  "extracted": 675,
  "first_members": [
    {
      "name": "zenodo/abcel_absoluteCN_30kbps_refitted.rds",
      "size": 357474067,
      "crc": 1034381081
    },
    {
      "name": "zenodo/britroc_30kb_ds_absCopyNumber.rds",
      "size": 160538274,
      "crc": 3031153864
    },
    {
      "name": "zenodo/britroc_smoothed_copyNumbersSegmented.rds",
      "size": 388320912,
      "crc": 362507276
    },
    {
      "name": "zenodo/BriTROC-1_short_variants.tar.gz",
      "size": 58956312,
      "crc": 1887820431
    },
    {
      "name": "zenodo/clonality_results/",
      "size": 0,
      "crc": 0
    },
    {
      "name": "zenodo/clonality_results/IM_10_clonality.csv",
      "size": 416561,
      "crc": 3965514222
    },
    {
      "name": "zenodo/clonality_results/IM_10_clonality.pdf",
      "size": 31908,
      "crc": 1598830394
    },
    {
      "name":

Identify BriTROC clinical and linkage files

In [5]:
from pathlib import Path
import json
p=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane1_britroc/zenodo_full_download.json')
d=json.loads(p.read_text())
for x in d['zip_members']:
 low=x['name'].lower()
 if any(k in low for k in ['clinical','metadata','sample','patient','manifest','phenotype','treatment','outcome']) or (not '/clonality_results/' in low and not low.endswith('.pdf')):
  print(x)


{'name': 'zenodo/abcel_absoluteCN_30kbps_refitted.rds', 'size': 357474067, 'crc': 1034381081}
{'name': 'zenodo/britroc_30kb_ds_absCopyNumber.rds', 'size': 160538274, 'crc': 3031153864}
{'name': 'zenodo/britroc_smoothed_copyNumbersSegmented.rds', 'size': 388320912, 'crc': 362507276}
{'name': 'zenodo/BriTROC-1_short_variants.tar.gz', 'size': 58956312, 'crc': 1887820431}
{'name': 'zenodo/component_parameters.rds', 'size': 92919500, 'crc': 953417557}
{'name': 'zenodo/prenorm_downsampled_fitsbritroc_30kb_ds_absCopyNumber.rds', 'size': 166642032, 'crc': 130713308}


Inspect BriTROC processed molecular object schemas

In [7]:
paths <- c(
  "/workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/britroc_30kb_ds_absCopyNumber.rds",
  "/workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/component_parameters.rds",
  "/workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/prenorm_downsampled_fitsbritroc_30kb_ds_absCopyNumber.rds"
)
inspect_obj <- function(p) {
  cat("\nFILE", p, "exists", file.exists(p), "\n")
  if (!file.exists(p)) return(NULL)
  x <- readRDS(p)
  cat("class:", paste(class(x),collapse=","), " dim:", paste(dim(x),collapse="x"), " length:", length(x), "\n")
  cat("names:", paste(head(names(x),100),collapse=" | "), "\n")
  if (is.data.frame(x)) cat("columns:", paste(colnames(x),collapse=" | "),"\n")
  if (is.list(x)) {
    for (nm in head(names(x),50)) {
      y <- x[[nm]]
      cat(" child",nm,"class",paste(class(y),collapse=","),"dim",paste(dim(y),collapse="x"),"length",length(y),"\n")
      if (is.data.frame(y) || is.matrix(y)) cat("  cols",paste(head(colnames(y),80),collapse=" | "),"\n")
    }
  }
  invisible(NULL)
}
lapply(paths, inspect_obj)


FILE /workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/britroc_30kb_ds_absCopyNumber.rds exists TRUE 
Loading required package: QDNAseqmod

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘QDNAseqmod’”
Error in .requirePackage(package): unable to find required package ‘QDNAseqmod’
Traceback:

1. FUN(X[[i]], ...)
2. cat("class:", paste(class(x), collapse = ","), " dim:", paste(dim(x), 
 .     collapse = "x"), " length:", length(x), "\n")
3. paste(dim(x), collapse = "x")
4. dim(x)
5. .extendsForS3(structure("QDNAseqCopyNumbers", package = "QDNAseqmod"))
6. extends(Class, maybe = FALSE)
7. getClassDef(class1)
8. .requirePackage(package)
9. stop(gettextf("unable to find required package %s", sQuote(package)), 
 .     domain = NA)
10. .handleSimpleError(function (cnd) 
  . {
  .     watcher$capture_plot_and_output()
  .     cnd <- sanitize_call(cnd)
  .     watcher$push(cnd)
  .     switch(on_error, c

Inspect BriTROC RDS attributes without unavailable package

In [9]:
paths <- c(
  "/workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/britroc_30kb_ds_absCopyNumber.rds",
  "/workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/component_parameters.rds",
  "/workspace/mechanism_fit_britroc/zenodo_extracted/zenodo/prenorm_downsampled_fitsbritroc_30kb_ds_absCopyNumber.rds"
)
for (p in paths) {
 cat("\nFILE",basename(p),"exists",file.exists(p),"\n")
 if (!file.exists(p)) next
 x <- tryCatch(readRDS(p), error=function(e)e)
 if (inherits(x,"error")) {cat("read error",conditionMessage(x),"\n"); next}
 at <- attributes(x)
 cat("oldClass",paste(oldClass(x),collapse=",")," typeof",typeof(x)," length",length(x),"\n")
 cat("attribute names",paste(names(at),collapse=" | "),"\n")
 if (!is.null(at$names)) cat("names",paste(head(at$names,100),collapse=" | "),"\n")
 if (is.list(x) && !is.null(at$names)) {
  for (i in seq_len(min(length(x),30))) {
   y<-x[[i]]; cat(" child",at$names[i],"typeof",typeof(y),"length",length(y),"attrs",paste(names(attributes(y)),collapse="|"),"\n")
   yn<-attributes(y)$names; if(!is.null(yn)) cat("  names",paste(head(yn,80),collapse=" | "),"\n")
  }
 }
 rm(x); gc()
}


FILE britroc_30kb_ds_absCopyNumber.rds exists TRUE 
Loading required package: QDNAseqmod

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘QDNAseqmod’”
Error in .requirePackage(package): unable to find required package ‘QDNAseqmod’
Traceback:

1. extends(Class, maybe = FALSE)
2. getClassDef(class1)
3. .requirePackage(package)
4. stop(gettextf("unable to find required package %s", sQuote(package)), 
 .     domain = NA)
5. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "unable to find required package ‘QDNAseqmod’", base::quote(.requirePackage(package)))

Verify corrected BriTROC EGA metadata endpoints

In [11]:
from pathlib import Path
import requests,json
from datetime import datetime,timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane1_britroc')
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
for eid,path in [('EGAS00001007292','studies'),('EGAD00001011049','datasets'),('EGAD00001011058','datasets'),('EGAC00001000388','dacs')]:
 url=f'https://metadata.ega-archive.org/{path}/{eid}'; at=ts(); r=requests.get(url,timeout=90)
 rec={'url':url,'timestamp':at,'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:10000]}
 (OUT/f'{eid}_public_metadata_correct_receipt.json').write_text(json.dumps(rec,indent=2))
 print(eid,r.status_code,r.text[:500])

EGAS00001007292 200 {"accession_id":"EGAS00001007292","title":"The genomic landscape of recurrent ovarian high grade serous carcinoma: the BriTROC-1 study","description":"The BriTROC-1 study, spanning 15 centres across the United Kingdom, is a prospective observational project investigating putative causes of relapse in high-grade serous carcinoma (HGSC). Potential genomic changes associated with disease recurrence are examined by studying paired tumour samples collected from patients at diagnosis and after relapse
EGAD00001011049 200 {"accession_id":"EGAD00001011049","title":"Shallow whole genome sequencing","description":"This dataset contains a collection of tumour samples from high grade serous ovarian carcinoma patients with recurrent disease collected near the point of diagnosis as well as tumour samples collected after patient relapse upon or some time after study entry. The majority of diagnosis samples were preserved in neutral buffered formalin whereas the majority of post-re

Cross-review PDS and Synapse access conclusions

In [13]:
from pathlib import Path
import json
from datetime import datetime,timezone
root=Path('/mnt/shared-workspace/shared/mechanism_fit_audit'); p=json.loads((root/'lane2_pds/lane2_manifest.json').read_text()); s=json.loads((root/'lane3_synapse/lane3_manifest_extended.json').read_text())
review={'reviewer_lane':'BriTROC worker cross-review','timestamp':datetime.now(timezone.utc).isoformat().replace('+00:00','Z'),'pds':{'live_catalog_records':p['catalog']['record_count'],'ovarian_packages':len(p['ovarian_packages']),'package_identity':'NCT00694382 / donation 119; mixed-cancer VTE-prevention trial, not ovarian-treatment efficacy','download_statuses':[f.get('status_code') for x in p['ovarian_packages'] for f in x['files'] if f.get('status_code') is not None],'verdict':'ACCESS_DENIED_OR_CONTROLLED'},'synapse':{'token_visible':s['authenticated_token_present'],'named_entities':len(s['entities']),'msk_children':s['msk_spectrum_children_discovered'],'direct_file_downloads':'403 for tested MSK processed WGS and TCGA-OV file handles','misidentified_id':'syn26642505 is an NSCLC PD-L1 project, not MSK-SPECTRUM clinical metadata; syn26642506 metadata returned 403','verdict':'ACCESS_DENIED_OR_CONTROLLED'},'decision':'Neither controlled lane contributes a currently eligible patient-linked benchmark.'}
(root/'cross_review_access.json').write_text(json.dumps(review,indent=2));print(json.dumps(review,indent=2))

{
  "reviewer_lane": "BriTROC worker cross-review",
  "timestamp": "2026-07-16T21:05:07.018349Z",
  "pds": {
    "live_catalog_records": 99,
    "ovarian_packages": 1,
    "package_identity": "NCT00694382 / donation 119; mixed-cancer VTE-prevention trial, not ovarian-treatment efficacy",
    "download_statuses": [
      401,
      401,
      401,
      401,
      401
    ],
    "verdict": "ACCESS_DENIED_OR_CONTROLLED"
  },
  "synapse": {
    "token_visible": false,
    "named_entities": 11,
    "msk_children": 2722,
    "direct_file_downloads": "403 for tested MSK processed WGS and TCGA-OV file handles",
    "misidentified_id": "syn26642505 is an NSCLC PD-L1 project, not MSK-SPECTRUM clinical metadata; syn26642506 metadata returned 403",
    "verdict": "ACCESS_DENIED_OR_CONTROLLED"
  },
  "decision": "Neither controlled lane contributes a currently eligible patient-linked benchmark."
}
